# 전체 대화 기록: LangGraph checkpointer

> 업데이트 기준: 2026-09 · LangChain 1.x / LangGraph 1.x

과거의 `ConversationBufferMemory`와 `ConversationChain` 대신, 현재는 대화를
**그래프 상태(state)의 `messages`** 로 표현하고 **checkpointer** 로 스레드별 상태를
저장하는 방식이 권장됩니다.

이 예제는 한 스레드의 전체 대화를 보존하고, 다음 호출에서 자동으로 다시 불러오는
가장 기본적인 단기 메모리를 만듭니다.

참고: [LangChain 단기 메모리](https://docs.langchain.com/oss/python/langchain/short-term-memory),
[LangGraph checkpointer](https://docs.langchain.com/oss/python/langgraph/checkpointers)


In [ ]:
# 필요한 경우 아래 줄의 주석을 해제하고 한 번만 실행하세요.
# %pip install -qU "langchain>=1.0" "langchain-openai>=1.0" "langgraph>=1.0" python-dotenv


In [3]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

# 다른 공급자를 쓸 때는 예: anthropic:claude-... 처럼 지정할 수 있습니다.
MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.6-luna")
model = init_chat_model(MODEL_ID)


## 메시지 상태를 가진 그래프 만들기

`MessagesState`의 `messages` 필드에는 메시지 병합 reducer가 이미 설정되어 있습니다.
노드는 새 AI 메시지만 반환하고, `InMemorySaver`가 `thread_id`별 전체 상태를 보존합니다.
`InMemorySaver`는 학습·테스트용이며, 운영 환경에서는 DB 기반 checkpointer를 사용합니다.


In [10]:
from langchain.messages import SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph

SYSTEM_PROMPT = "당신은 친절하고 정확한 은행 상담 챗봇입니다."


def call_model(state: MessagesState):
    response = model.invoke(
        [SystemMessage(content=SYSTEM_PROMPT), *state["messages"]]
    )
    # MessagesState의 reducer가 기존 목록 뒤에 이 메시지를 추가합니다.
    return {"messages": [response]}


builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")
builder.add_edge("model", END)

chat = builder.compile(checkpointer=InMemorySaver())


## 같은 스레드에서 대화 이어가기

메모리의 키는 `thread_id`입니다. 같은 값을 사용하면 직전 호출의 상태가 자동으로
복원됩니다. 매 호출에는 **새 사용자 메시지만** 전달합니다.


In [11]:
config = {"configurable": {"thread_id": "bank-account-demo"}}


def ask(text: str) -> str:
    result = chat.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config=config,
    )
    return result["messages"][-1].content


In [6]:
print(
    ask(
        "안녕하세요. 비대면으로 은행 계좌를 개설하고 싶습니다. "
        "어떻게 시작해야 하나요?"
    )
)


안녕하세요. 비대면 계좌 개설은 보통 은행 앱에서 다음 순서로 진행합니다.

1. **은행 공식 앱 설치**
   - 앱스토어·구글플레이에서 은행명을 직접 검색해 공식 앱을 설치하세요.
   - 문자나 광고 링크를 통한 설치는 피하는 것이 안전합니다.

2. **입출금 계좌 개설 메뉴 선택**
   - 앱에서 `상품가입` 또는 `계좌개설` → `입출금 통장`을 선택합니다.

3. **본인확인 진행**
   - 휴대폰 본인인증
   - 신분증 촬영(주민등록증 또는 운전면허증)
   - 얼굴 인증 또는 영상통화
   - 다른 은행 계좌를 이용한 추가 인증 등이 필요할 수 있습니다.

4. **고객정보 및 거래 목적 입력**
   - 주소, 직업, 거래 목적, 자금 출처 등을 입력합니다.
   - 금융거래 목적 확인 절차에 따라 한도제한계좌로 개설될 수도 있습니다.

5. **약관 동의 및 비밀번호 설정**
   - 상품 설명과 약관을 확인한 뒤 계좌 비밀번호를 설정하면 개설이 완료됩니다.

**미리 준비할 것**
- 본인 명의 휴대전화
- 유효한 주민등록증 또는 운전면허증
- 본인 명의의 다른 은행 계좌(필요 시)
- 안정적인 인터넷 환경

은행별로 나이·국적·체류자격, 기존 계좌 보유 여부에 따라 절차가 다를 수 있습니다. 또한 신분증 사진, 인증번호, 보안카드·OTP 정보는 누구에게도 보내지 마세요. 특정 은행에서 개설하려는 경우 은행명을 알려주시면 해당 절차에 맞춰 안내해 드리겠습니다.


In [7]:
print(ask("방금 안내한 절차를 불렛포인트로 다시 정리해 주세요."))


- **은행 공식 앱 설치**
  - 앱스토어·구글플레이에서 은행명을 직접 검색해 설치
  - 문자나 광고 링크를 통한 설치는 피하기

- **계좌 개설 메뉴 선택**
  - 앱에서 `상품가입` 또는 `계좌개설` 선택
  - 원하는 입출금 통장 상품 선택

- **본인확인 진행**
  - 휴대폰 본인인증
  - 주민등록증 또는 운전면허증 촬영
  - 얼굴 인증 또는 영상통화
  - 필요 시 다른 은행 계좌로 추가 인증

- **고객정보 및 거래 목적 입력**
  - 주소, 직업, 거래 목적, 자금 출처 등 입력
  - 심사 결과에 따라 한도제한계좌로 개설될 수 있음

- **약관 동의 및 비밀번호 설정**
  - 상품 설명과 약관 확인
  - 계좌 비밀번호 설정 후 개설 완료

- **미리 준비할 것**
  - 본인 명의 휴대전화
  - 유효한 주민등록증 또는 운전면허증
  - 본인 명의의 다른 은행 계좌(필요 시)
  - 안정적인 인터넷 환경

- **주의사항**
  - 은행별로 가입 조건과 절차가 다를 수 있음
  - 신분증 사진, 인증번호, 보안카드·OTP 정보는 타인에게 제공하지 않기


## 저장된 상태 확인

레거시 API의 `load_memory_variables()` 대신 `get_state()`로 최신 checkpoint를 읽습니다.


In [8]:
snapshot = chat.get_state(config)

for message in snapshot.values["messages"]:
    print(f"[{message.type}] {message.content}\n")


[human] 안녕하세요. 비대면으로 은행 계좌를 개설하고 싶습니다. 어떻게 시작해야 하나요?

[ai] 안녕하세요. 비대면 계좌 개설은 보통 은행 앱에서 다음 순서로 진행합니다.

1. **은행 공식 앱 설치**
   - 앱스토어·구글플레이에서 은행명을 직접 검색해 공식 앱을 설치하세요.
   - 문자나 광고 링크를 통한 설치는 피하는 것이 안전합니다.

2. **입출금 계좌 개설 메뉴 선택**
   - 앱에서 `상품가입` 또는 `계좌개설` → `입출금 통장`을 선택합니다.

3. **본인확인 진행**
   - 휴대폰 본인인증
   - 신분증 촬영(주민등록증 또는 운전면허증)
   - 얼굴 인증 또는 영상통화
   - 다른 은행 계좌를 이용한 추가 인증 등이 필요할 수 있습니다.

4. **고객정보 및 거래 목적 입력**
   - 주소, 직업, 거래 목적, 자금 출처 등을 입력합니다.
   - 금융거래 목적 확인 절차에 따라 한도제한계좌로 개설될 수도 있습니다.

5. **약관 동의 및 비밀번호 설정**
   - 상품 설명과 약관을 확인한 뒤 계좌 비밀번호를 설정하면 개설이 완료됩니다.

**미리 준비할 것**
- 본인 명의 휴대전화
- 유효한 주민등록증 또는 운전면허증
- 본인 명의의 다른 은행 계좌(필요 시)
- 안정적인 인터넷 환경

은행별로 나이·국적·체류자격, 기존 계좌 보유 여부에 따라 절차가 다를 수 있습니다. 또한 신분증 사진, 인증번호, 보안카드·OTP 정보는 누구에게도 보내지 마세요. 특정 은행에서 개설하려는 경우 은행명을 알려주시면 해당 절차에 맞춰 안내해 드리겠습니다.

[human] 방금 안내한 절차를 불렛포인트로 다시 정리해 주세요.

[ai] - **은행 공식 앱 설치**
  - 앱스토어·구글플레이에서 은행명을 직접 검색해 설치
  - 문자나 광고 링크를 통한 설치는 피하기

- **계좌 개설 메뉴 선택**
  - 앱에서 `상품가입` 또는 `계좌개설` 선택
  - 원하는 입출금 통장 상품 선택

- **본인확인 진행**
  - 휴대

## 스레드 격리 확인

다른 `thread_id`는 별도의 대화이므로 앞선 내용을 알지 못합니다.


In [9]:
other_config = {"configurable": {"thread_id": "another-customer"}}
result = chat.invoke(
    {"messages": [{"role": "user", "content": "제가 방금 무엇을 물었나요?"}]},
    config=other_config,
)
print(result["messages"][-1].content)


방금 “제가 방금 무엇을 물었나요?”라고 물으셨습니다.
